<a href="https://colab.research.google.com/github/Lyv-ux/DI_Bootcamp/blob/main/W6D5_DailyChallenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================================
# DAILY CHALLENGE: BUILDING TRUSTWORTHY INSIGHTS WITH BERT
# ==========================================================

# ----------------------------------------------------------
# 1. Install and import libraries
# ----------------------------------------------------------
!pip -q install datasets transformers evaluate scikit-learn seaborn matplotlib

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

import evaluate
from sklearn.metrics import accuracy_score, f1_score
from scipy.special import softmax

# ----------------------------------------------------------
# 2. Load tweet_eval sentiment dataset
# ----------------------------------------------------------
dataset = load_dataset("tweet_eval", "sentiment")

print(dataset)

# ----------------------------------------------------------
# 3. Inspect class distribution
# Labels:
# 0 = Negative
# 1 = Neutral
# 2 = Positive
# ----------------------------------------------------------
label_names = {
    0: "Negative",
    1: "Neutral",
    2: "Positive"
}

print("\nClass distribution (train):")
train_labels = dataset["train"]["label"]

for label in sorted(set(train_labels)):
    print(f"{label_names[label]}: {train_labels.count(label)}")

# ----------------------------------------------------------
# 4. Save two example tweets per label
# ----------------------------------------------------------
examples = {}

for label in [0, 1, 2]:
    tweets = []

    for item in dataset["train"]:
        if item["label"] == label:
            tweets.append(item["text"])

        if len(tweets) == 2:
            break

    examples[label] = tweets

print("\nExamples:")
for label, tweets in examples.items():
    print(f"\n{label_names[label]}")
    for tweet in tweets:
        print("-", tweet)

# ----------------------------------------------------------
# 5. Initialize DistilBERT tokenizer
# ----------------------------------------------------------
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ----------------------------------------------------------
# 6. Preprocessing function
# Truncate/pad to 128 tokens
# ----------------------------------------------------------
def preprocess(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_dataset = dataset.map(
    preprocess,
    batched=True
)

# ----------------------------------------------------------
# 7. Rename label column and set torch format
# ----------------------------------------------------------
tokenized_dataset = tokenized_dataset.rename_column(
    "label",
    "labels"
)

tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

# ----------------------------------------------------------
# 8. Load DistilBERT classification model
# ----------------------------------------------------------
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
)

# ----------------------------------------------------------
# 9. Metrics function
# Accuracy + Macro F1
# ----------------------------------------------------------
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_metric.compute(
        predictions=predictions,
        references=labels
    )

    f1 = f1_metric.compute(
        predictions=predictions,
        references=labels,
        average="macro"
    )

    return {
        "accuracy": accuracy["accuracy"],
        "macro_f1": f1["f1"]
    }

# ----------------------------------------------------------
# 10. Training arguments
# ----------------------------------------------------------
training_args = TrainingArguments(
    output_dir="./distilbert_sentiment",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=100
)

# ----------------------------------------------------------
# 11. Trainer
# ----------------------------------------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics
)

# ----------------------------------------------------------
# 12. Train model
# ----------------------------------------------------------
trainer.train()

# ----------------------------------------------------------
# 13. Save best checkpoint
# ----------------------------------------------------------
trainer.save_model("./saved_distilbert_sentiment")
tokenizer.save_pretrained("./saved_distilbert_sentiment")

# ----------------------------------------------------------
# 14. Validation evaluation
# ----------------------------------------------------------
results = trainer.evaluate()

print("\nValidation Results:")
print(results)

# ----------------------------------------------------------
# 15. Test set confidence scores
# ----------------------------------------------------------
predictions = trainer.predict(
    tokenized_dataset["test"]
)

logits = predictions.predictions

probabilities = softmax(logits, axis=1)

confidence_scores = probabilities.max(axis=1)

# ----------------------------------------------------------
# 16. Plot histogram of confidence scores
# ----------------------------------------------------------
plt.figure(figsize=(8,5))

plt.hist(
    confidence_scores,
    bins=np.arange(0, 1.1, 0.1),
    edgecolor="black"
)

plt.title("Confidence Score Distribution")
plt.xlabel("Confidence")
plt.ylabel("Number of Predictions")
plt.show()

print("""
Interpretation:
- Many predictions near 1.0 suggest high confidence.
- Too many incorrect high-confidence predictions may indicate overconfidence.
- A broader distribution may indicate better calibration.
""")

# ----------------------------------------------------------
# 17. Attention inspection
# ----------------------------------------------------------
attention_model = AutoModel.from_pretrained(
    MODEL_NAME,
    output_attentions=True
)

example_tweet = examples[0][0]

print("\nSelected Tweet:")
print(example_tweet)

encoded = tokenizer(
    example_tweet,
    return_tensors="pt",
    truncation=True,
    max_length=128
)

with torch.no_grad():
    outputs = attention_model(**encoded)

# ----------------------------------------------------------
# 18. Last layer attention
# ----------------------------------------------------------
last_layer_attention = outputs.attentions[-1]

# Average heads
avg_attention = last_layer_attention.mean(dim=1).squeeze()

# CLS attention
cls_attention = avg_attention[0].cpu().numpy()

tokens = tokenizer.convert_ids_to_tokens(
    encoded["input_ids"][0]
)

# ----------------------------------------------------------
# 19. Attention visualization
# ----------------------------------------------------------
plt.figure(figsize=(12,5))

sns.barplot(
    x=tokens,
    y=cls_attention
)

plt.xticks(rotation=90)
plt.title("[CLS] Attention to Tokens")
plt.show()

# ----------------------------------------------------------
# 20. Top attended tokens
# ----------------------------------------------------------
attention_pairs = sorted(
    zip(tokens, cls_attention),
    key=lambda x: x[1],
    reverse=True
)

important_tokens = [
    token
    for token, score in attention_pairs[:5]
]

print("\nMost attended tokens:")
print(important_tokens)

# ----------------------------------------------------------
# 21. Explainable inference helper
# ----------------------------------------------------------
def analyze_text(text):

    encoded = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    with torch.no_grad():

        clf_output = model(**encoded)

        probs = torch.softmax(
            clf_output.logits,
            dim=1
        )[0]

        pred = torch.argmax(probs).item()

        confidence = probs[pred].item()

        attention_output = attention_model(**encoded)

    attention = (
        attention_output.attentions[-1]
        .mean(dim=1)
        .squeeze()[0]
        .cpu()
        .numpy()
    )

    tokens = tokenizer.convert_ids_to_tokens(
        encoded["input_ids"][0]
    )

    ranked = sorted(
        zip(tokens, attention),
        key=lambda x: x[1],
        reverse=True
    )

    highlighted_tokens = [
        token
        for token, score in ranked[:5]
    ]

    return {
        "label": label_names[pred],
        "confidence": round(confidence, 4),
        "highlighted_tokens": highlighted_tokens
    }

# ----------------------------------------------------------
# 22. Test inference helper
# ----------------------------------------------------------
result = analyze_text(
    "The customer service was excellent and solved my issue quickly."
)

print("\nInference Result:")
print(result)